In [1]:
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import os
from pathlib import Path
import polars.selectors as cs

In [2]:
input_folder=Path().resolve().parent /'input'
input_folder

PosixPath('/home/ahmed/MyOwnProjects/load-profile-decomposition/src/input')

In [3]:
pl.enable_string_cache()
pl.Config.set_streaming_chunk_size(2000000)

polars.config.Config

Building customer demographics behaviour

In [4]:
%%time
data=pl.scan_parquet(input_folder/'load_profile_buildingID_*')
BUILDING=data.collect().select(pl.col('bldg_id')).unique()

CPU times: user 616 ms, sys: 688 ms, total: 1.3 s
Wall time: 453 ms


## The aim of this section is to find as many features as possible to build accurate hypothesis

In [5]:
Meta_data=pl.scan_parquet(input_folder/'Meta_Data.parquet').select(pl.col('in.occupants','in.state','in.county',
                        'in.representative_income','in.area_median_income','in.income',
                        'in.income_recs_2020','in.income_recs_2015', 'in.building_america_climate_zone',
                        'in.bedrooms','in.tenure','in.household_has_tribal_persons')
                ).unique()

## First Hypothesis

the energy consumption patterns is significantly determined by the temporal attributes

## Data Preprocessing

In order to reach the desired objective we need to break the datetime into several columns each of which determines

In [6]:
df_hp1=data.with_columns(
            pl.col('timestamp').dt.weekday().alias('day of the week'),
            pl.col('timestamp').dt.hour().alias('hour of the day'),
            pl.col('timestamp').dt.day().alias('day of the month'),
            pl.col('timestamp').dt.ordinal_day().alias('day of the year'),
            pl.col('timestamp').dt.week().alias('week of the year'),
            pl.col('timestamp').dt.month().alias('month of the year'),
            pl.col('timestamp').dt.quarter().alias('quarter')).with_columns(
            pl.when(pl.col('day of the week').is_in([6,7])).then(0).otherwise(1).alias('IsWeekend')
            ).select(cs.matches('^out.electricity.*|^bldg*|^day*|^hour*|^week*|^month*|^time*|^quarter|^IsWeekend')).collect()

* Checking missing values

In [7]:
df_hp1.null_count()

bldg_id,timestamp,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,out.electricity.heating.energy_consumption..kwh,out.electricity.heating_fans_pumps.energy_consumption..kwh,out.electricity.heating_hp_bkup.energy_consumption..kwh,out.electricity.heating_hp_bkup_fa.energy_consumption..kwh,out.electricity.hot_water.energy_consumption..kwh,out.electricity.hot_water_solar_th.energy_consumption..kwh,out.electricity.lighting_exterior.energy_consumption..kwh,out.electricity.lighting_garage.energy_consumption..kwh,out.electricity.lighting_interior.energy_consumption..kwh,out.electricity.mech_vent.energy_consumption..kwh,out.electricity.net.energy_consumption..kwh,out.electricity.permanent_spa_heat.energy_consumption..kwh,out.electricity.permanent_spa_pump.energy_consumption..kwh,out.electricity.plug_loads.energy_consumption..kwh,out.electricity.pool_heater.energy_consumption..kwh,out.electricity.pool_pump.energy_consumption..kwh,out.electricity.pv.energy_consumption..kwh,out.electricity.range_oven.energy_consumption..kwh,out.electricity.refrigerator.energy_consumption..kwh,out.electricity.television.energy_consumption..kwh,out.electricity.total.energy_consumption..kwh,out.electricity.well_pump.energy_consumption..kwh,out.electricity.ceiling_fan.energy_consumption_intensity..kwh_per_ft2,out.electricity.clothes_dryer.energy_consumption_intensity..kwh_per_ft2,out.electricity.clothes_washer.energy_consumption_intensity..kwh_per_ft2,out.electricity.cooling.energy_consumption_intensity..kwh_per_ft2,out.electricity.cooling_fans_pumps.energy_consumption_intensity..kwh_per_ft2,out.electricity.dishwasher.energy_consumption_intensity..kwh_per_ft2,out.electricity.ev_charging.energy_consumption_intensity..kwh_per_ft2,out.electricity.freezer.energy_consumption_intensity..kwh_per_ft2,out.electricity.heating.energy_consumption_intensity..kwh_per_ft2,out.electricity.heating_fans_pumps.energy_consumption_intensity..kwh_per_ft2,out.electricity.heating_hp_bkup.energy_consumption_intensity..kwh_per_ft2,out.electricity.heating_hp_bkup_fa.energy_consumption_intensity..kwh_per_ft2,out.electricity.hot_water.energy_consumption_intensity..kwh_per_ft2,out.electricity.hot_water_solar_th.energy_consumption_intensity..kwh_per_ft2,out.electricity.lighting_exterior.energy_consumption_intensity..kwh_per_ft2,out.electricity.lighting_garage.energy_consumption_intensity..kwh_per_ft2,out.electricity.lighting_interior.energy_consumption_intensity..kwh_per_ft2,out.electricity.mech_vent.energy_consumption_intensity..kwh_per_ft2,out.electricity.net.energy_consumption_intensity..kwh_per_ft2,out.electricity.permanent_spa_heat.energy_consumption_intensity..kwh_per_ft2,out.electricity.permanent_spa_pump.energy_consumption_intensity..kwh_per_ft2,out.electricity.plug_loads.energy_consumption_intensity..kwh_per_ft2,out.electricity.pool_heater.energy_consumption_intensity..kwh_per_ft2,out.electricity.pool_pump.energy_consumption_intensity..kwh_per_ft2,out.electricity.pv.energy_consumption_intensity..kwh_per_ft2,out.electricity.range_oven.energy_consumption_intensity..kwh_per_ft2,out.electricity.refrigerator.energy_consumption_intensity..kwh_per_ft2,out.electricity.television.energy_consumption_intensity..kwh_per_ft2,out.electricity.total.energy_consumption_intensity..kwh_per_ft2,out.electricity.well_pump.energy_consumption_intensity..kwh_per_ft2,day of the week,hour of the day,day of the month,day of the year,week of the year,month of the year,quarter,IsWeekend
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u3

No missing values within the data

In [32]:
import importlib
import lib
from lib import loadProfile
importlib.reload(lib)
vis=loadProfile(df_hp1)

In [33]:
# Run boxplots
vis.boxplot_exp()

NameError: name 'dev_count' is not defined

In [34]:
vis.hist_exp()

NameError: name 'dev_count' is not defined

In [ ]:
plt.style.use('ggplot')
cols=[n for n in df_hp1.columns if n.startswith('out.electricity')]
fig, ax=plt.subplots(len(cols), 1, figsize=(10, 7 * len(cols)))
for i,x in enumerate(cols):
        sns.boxenplot(data=df_hp1, x=x, ax=ax[i])

### Now we want to see the total amount spent if its holiday or not

In [ ]:
sns.barplot(data=df_hp1, x='out.electricity.total.energy_consumption..kwh', y='IsWeekend')

### Objective 1: the hour of the day singificantly impact the consumption pattern

### Objective 2: the day of the month singificantly impact the consumption pattern

### Objective 3: the week of the month singificantly impact the consumption pattern

### Objective 4: the month of the year singificantly impact the consumption pattern

## Second Hypothesis

net total energy consumption determines the peak load periods

## Third Hypothesis

Simultaneous operation of two appliance creates a load on user's side

## Fourth Hypothesis

User Behaviour analysis